In [4]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/Intro to CV/skin_cancer.zip" /content/

!unzip -q /content/skin_cancer.zip -d /content/dataset/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
import torch.nn.functional as F

# 1. Device, Path, and Epoch Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_dir = '/content/dataset'
NUM_EPOCHS = 15  # Change this single variable to adjust the training cycle

# Dynamically locate the Train/Test folders
for root, dirs, files in os.walk(base_dir):
    if 'Train' in dirs and 'Test' in dirs:
        base_dir = root
        break

# 2. Data Preprocessing (ImageNet Standards)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(os.path.join(base_dir, 'Train'), transform=transform)
test_data = datasets.ImageFolder(os.path.join(base_dir, 'Test'), transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
num_classes = len(train_data.classes)

print(f"Loaded {num_classes} classes on {device}. Training for {NUM_EPOCHS} epochs per model.")

# 3. Model Dictionary Setup
models_dict = {
    "AlexNet": models.alexnet(weights=models.AlexNet_Weights.DEFAULT),
    "VGG16": models.vgg16(weights=models.VGG16_Weights.DEFAULT),
    "VGG19": models.vgg19(weights=models.VGG19_Weights.DEFAULT),
    "ResNet18": models.resnet18(weights=models.ResNet18_Weights.DEFAULT),
    "ResNet50": models.resnet50(weights=models.ResNet50_Weights.DEFAULT),
    "ResNet101": models.resnet101(weights=models.ResNet101_Weights.DEFAULT),
    "DenseNet121": models.densenet121(weights=models.DenseNet121_Weights.DEFAULT),
    "EfficientNet-B0": models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
}

# 4. Unified Training and Evaluation Loop
for name, model in models_dict.items():
    print(f"\n{'='*40}")
    print(f"--- Initializing {name} ---")

    # Dynamically replace the final layer
    if "AlexNet" in name or "VGG" in name:
        model.classifier[6] = nn.Linear(4096, num_classes)
    elif "ResNet" in name:
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif "DenseNet" in name:
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif "EfficientNet" in name:
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    # Epoch Cycle
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"[{name}] Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {running_loss/len(train_loader):.4f}")

    # 5. Evaluation Phase
    print(f"Evaluating {name} on Test Set...")
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # Metrics Calculation for Table 1
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
    auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')

    print(f"\n--- Final Metrics for {name} ---")
    print(f"Accuracy:  {accuracy*100:.2f}%")
    print(f"Precision: {precision*100:.2f}%")
    print(f"Recall:    {recall*100:.2f}%")
    print(f"F1-Score:  {f1*100:.2f}%")
    print(f"AUC:       {auc*100:.2f}%")

Loaded 9 classes on cuda. Training for 15 epochs per model.

--- Initializing AlexNet ---
[AlexNet] Epoch 1/15 - Loss: 1.4361
[AlexNet] Epoch 2/15 - Loss: 0.9692
[AlexNet] Epoch 3/15 - Loss: 0.7268
[AlexNet] Epoch 4/15 - Loss: 0.5519
[AlexNet] Epoch 5/15 - Loss: 0.4322
[AlexNet] Epoch 6/15 - Loss: 0.3231
[AlexNet] Epoch 7/15 - Loss: 0.2802
[AlexNet] Epoch 8/15 - Loss: 0.2493
[AlexNet] Epoch 9/15 - Loss: 0.1994
[AlexNet] Epoch 10/15 - Loss: 0.1743
[AlexNet] Epoch 11/15 - Loss: 0.1886
[AlexNet] Epoch 12/15 - Loss: 0.1964
[AlexNet] Epoch 13/15 - Loss: 0.1679
[AlexNet] Epoch 14/15 - Loss: 0.1519
[AlexNet] Epoch 15/15 - Loss: 0.1483
Evaluating AlexNet on Test Set...

--- Final Metrics for AlexNet ---
Accuracy:  48.31%
Precision: 45.74%
Recall:    48.61%
F1-Score:  43.72%
AUC:       89.32%

--- Initializing VGG16 ---
[VGG16] Epoch 1/15 - Loss: 1.5751
[VGG16] Epoch 2/15 - Loss: 1.0560
[VGG16] Epoch 3/15 - Loss: 0.8584
[VGG16] Epoch 4/15 - Loss: 0.6800
[VGG16] Epoch 5/15 - Loss: 0.5347
[VGG16]

In [ ]:
!pip install thop

In [9]:
import torch
import torch.nn as nn
from torchvision import models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np

# 1. Initialize Feature Extractor (ResNet50)
print("Setting up ResNet50 Feature Extractor...")
feature_extractor = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
feature_extractor.fc = nn.Identity()  # Remove final layer to get the 2048-d feature vector
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

# 2. Function to Extract Features
def get_features(dataloader):
    features, labels = [], []
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            outputs = feature_extractor(inputs)
            features.append(outputs.cpu().numpy())
            labels.append(targets.numpy())
    return np.vstack(features), np.concatenate(labels)

print("Extracting training features (this may take a few minutes)...")
X_train, y_train = get_features(train_loader)
print("Extracting testing features...")
X_test, y_test = get_features(test_loader)

# 3. Define Classifiers for Table 2
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(),
    "Linear SVM": SVC(kernel='linear', probability=True),
    "RBF-SVM": SVC(kernel='rbf', probability=True),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

# 4. Train and Evaluate Each Classifier
print("\n--- Training Classifiers for Table 2 ---")
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)

    # Predictions
    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)

    # Metrics
    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
    auc = roc_auc_score(y_test, probs, multi_class='ovr')

    print(f"\n{name}:")
    print(f"Accuracy: {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}% | AUC: {auc*100:.2f}%")

Setting up ResNet50 Feature Extractor...
Extracting training features (this may take a few minutes)...
Extracting testing features...

--- Training Classifiers for Table 2 ---

Logistic Regression:
Accuracy: 46.61% | Precision: 48.30% | Recall: 47.22% | F1: 44.21% | AUC: 87.63%

Decision Tree:
Accuracy: 24.58% | Precision: 22.47% | Recall: 29.17% | F1: 23.88% | AUC: 59.62%

Random Forest:
Accuracy: 35.59% | Precision: 44.45% | Recall: 38.19% | F1: 30.85% | AUC: 77.51%

K-Nearest Neighbors (KNN):
Accuracy: 37.29% | Precision: 37.65% | Recall: 36.57% | F1: 33.59% | AUC: 75.93%

Linear SVM:
Accuracy: 44.07% | Precision: 41.68% | Recall: 45.14% | F1: 40.09% | AUC: 89.11%

RBF-SVM:
Accuracy: 45.76% | Precision: 51.74% | Recall: 46.53% | F1: 43.23% | AUC: 89.03%


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [06:56:14] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost:
Accuracy: 47.46% | Precision: 48.10% | Recall: 44.91% | F1: 42.73% | AUC: 79.77%


In [10]:
import torch
import os
import time
from torchvision import models
from thop import profile # Ensure !pip install thop was run

# Re-instantiate standard models for baseline profiling
profiling_models = {
    "AlexNet": models.alexnet(),
    "VGG16": models.vgg16(),
    "VGG19": models.vgg19(),
    "ResNet18": models.resnet18(),
    "ResNet50": models.resnet50(),
    "ResNet101": models.resnet101(),
    "DenseNet121": models.densenet121(),
    "EfficientNet-B0": models.efficientnet_b0()
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dummy_input = torch.randn(1, 3, 224, 224).to(device)

print(f"{'Model':<15} | {'Params (M)':<12} | {'Size (MB)':<12} | {'FLOPs (G)':<12} | {'Inference (ms)':<15}")
print("-" * 75)

for name, model in profiling_models.items():
    model = model.to(device)
    model.eval()

    # 1. Parameters (Millions)
    params = sum(p.numel() for p in model.parameters()) / 1e6

    # 2. Model Size (MB)
    torch.save(model.state_dict(), 'temp.p')
    size_mb = os.path.getsize('temp.p') / (1024 * 1024)
    os.remove('temp.p')

    # 3. FLOPs (GigaFLOPs) using THOP
    macs, _ = profile(model, inputs=(dummy_input, ), verbose=False)
    flops_g = (macs * 2) / 1e9 # 1 MAC = ~2 FLOPs

    # 4. Inference Time (ms) - Averaged over 100 runs for stability
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy_input)

    start_time = time.time()
    with torch.no_grad():
        for _ in range(100):
            _ = model(dummy_input)
    # Convert total seconds to average milliseconds per image
    infer_time = ((time.time() - start_time) / 100) * 1000

    print(f"{name:<15} | {params:<12.2f} | {size_mb:<12.2f} | {flops_g:<12.2f} | {infer_time:<15.2f}")

Model           | Params (M)   | Size (MB)    | FLOPs (G)    | Inference (ms) 
---------------------------------------------------------------------------
AlexNet         | 61.10        | 233.09       | 1.43         | 1.57           
VGG16           | 138.36       | 527.80       | 30.94        | 9.14           
VGG19           | 143.67       | 548.06       | 39.26        | 11.34          
ResNet18        | 11.69        | 44.66        | 3.65         | 2.24           
ResNet50        | 25.56        | 97.78        | 8.27         | 5.95           
ResNet101       | 44.55        | 170.51       | 15.73        | 12.78          
DenseNet121     | 7.98         | 30.97        | 5.79         | 20.92          
EfficientNet-B0 | 5.29         | 20.44        | 0.83         | 9.69           
